In [7]:
import openai, json, httpx

client = openai.OpenAI()
messages = []

In [8]:
BASE_URL = "https://nomad-movies-2.nomadcoders.workers.dev"

def get_popular_movies():
    return httpx.get(f"{BASE_URL}/movies").json()

def get_movies_details(id):
    return httpx.get(f"{BASE_URL}/movies/{id}").json()

def get_movies_credits(id):
    return httpx.get(f"{BASE_URL}/movies/{id}/credits").json()

def get_similar_movies(id):
    return httpx.get(f"{BASE_URL}/movies/{id}/similar").json()

FUNCTIONS_MAP = {
    "get_popular_movies": get_popular_movies,
    "get_movies_details": get_movies_details,
    "get_movies_credits": get_movies_credits,
    "get_similar_movies": get_similar_movies
}

In [9]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "A function to get the popular movies",
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_movies_details",
            "description": "A function to get the details of a movie",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "string",
                        "description": "The id of the movie to get the details of"
                    }
                },
                "required": ["id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_movies_credits",
            "description": "A function to get the credits of a movie",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "string",
                        "description": "The id of the movie to get the credits of"
                    }
                },
                "required": ["id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_similar_movies",
            "description": "A function to get similar movies",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "string",
                        "description": "The id of the movie to get the similar of"
                    }
                },
                "required": ["id"]
            }
        }
    },
]

In [10]:
from openai.types.chat import ChatCompletionMessage

def process_ai_response(message: ChatCompletionMessage):
    if message.tool_calls:
        messages.append({
            "role": "assistant",
            "content": message.content or "",
            "tool_calls": [
                {
                    "id": tool_call.id,
                    "type": "function",
                    "function": {
                        "name": tool_call.function.name,
                        "arguments": tool_call.function.arguments
                    }
                } for tool_call in message.tool_calls
            ]
        })

        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            function_args = tool_call.function.arguments
            
            try:
                arguments = json.loads(function_args)
            except json.JSONDecodeError:
                arguments = {}
            
            function_to_run = FUNCTIONS_MAP.get(function_name)

            result = function_to_run(**arguments)

            print(f"Agent: {function_name} 호출")

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": function_name,
                "content": json.dumps(result)
            })
        
        call_ai()
    else:
        messages.append({
            "role": "assistant",
            "content": message.content
        })
        print(f"Agent: {message.content}")

def call_ai():
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=TOOLS,
    )
    process_ai_response(response.choices[0].message)

In [11]:
while True:
    message = input("Send a message to the LLM ...")
    if message == "quit" or message == "q":
        break
    else:
        messages.append({
            "role": "user",
            "content": message
        })
        print(f"User: {message}")
        call_ai()

User: 지금 인기 있는 영화 알려줘
Agent: get_popular_movies 호출
Agent: 지금 인기 있는 영화 목록은 다음과 같습니다:

1. **Obsession**
   - 개요: "One Wish Willow"를 깨뜨리고 사랑하는 사람을 얻으려는 한 남자의 어두운 가격을 발견하는 이야기.
   - 출시일: 2026년 5월 13일
   - 평점: 7.9
   - ![Obsession](https://image.tmdb.org/t/p/w780/bRwnj8WEKBCvmfeUNOukJPwB43K.jpg)

2. **Peddi**
   - 개요: 1980년대 안드라프라데시의 한 마을 주민이 스포츠를 통해 공동체를 단합시켜 강력한 적에 맞서 싸우는 이야기.
   - 출시일: 2026년 6월 3일
   - 평점: 6.5
   - ![Peddi](https://image.tmdb.org/t/p/w780/kJAJNNBYlbqAcpTDxBNnaILSMTy.jpg)

3. **Hai Jawani Toh Ishq Hona Hai**
   - 개요: 결혼을 포기한 주인공이 새로운 외국에서의 사랑과 충격적인 사실을 마주하는 이야기.
   - 출시일: 2026년 6월 4일
   - 평점: 5.5
   - ![Hai Jawani Toh Ishq Hona Hai](https://image.tmdb.org/t/p/w780/vmlJvz6qVzYgei2V74GvnmcuQfW.jpg)

4. **The Unknown Man**
   - 개요: 영감을 찾기 위해 프랑스 리비에라의 한 고립된 장소로 간 벨기에 작가의 이야기.
   - 출시일: 2021년 10월 16일
   - 평점: 8.2
   - ![The Unknown Man](https://image.tmdb.org/t/p/w780/4TpBhdaSl5ALHbgeaYOLF8Q3haz.jpg)

5. **Mortal Kombat II**
   - 개요: 악의 군주인 샤오 칸과 그의 적들에 맞서 싸우는 챔피언들의 이야기.
   -